In [9]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import dgl
from scipy.stats import entropy

DEVICE = torch.device("cpu")

def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

seed_all(42)

DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")
OUT.mkdir(parents=True, exist_ok=True)


In [10]:
ck = torch.load(DATA, weights_only=False)
g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]

g = g.to(DEVICE)
feats = feats.to(DEVICE)
labels = labels.to(DEVICE)

N, d = feats.shape
C = int(labels.max() + 1)

t = torch.from_numpy(np.load(OUT / "teacher_logits.npz")["logits"]).to(DEVICE)
print(f"teacher {t.shape} N={N} E={g.num_edges()}")

def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


teacher torch.Size([13752, 10]) N=13752 E=505474


In [11]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(d, 512)
        self.fc2 = nn.Linear(512, C)
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.drop(h)
        return self.fc2(h)

def hardness_scores(pt_logits, ps_logits, edge_list, alpha):
    pt = pt_logits.softmax(-1).detach().cpu().numpy()
    ps = ps_logits.softmax(-1).detach().cpu().numpy()
    Ht = entropy(pt, axis=1)
    Hs = entropy(ps, axis=1)
    Ht /= Ht.max() + 1e-12
    Hs /= Hs.max() + 1e-12
    Ht = np.clip(Ht, 1e-8, None)
    Hs = np.clip(Hs, 1e-8, None)
    s, dst = edge_list[0].numpy(), edge_list[1].numpy()
    dot = (pt[s] * pt[dst]).sum(-1)
    norm = np.linalg.norm(pt[s], axis=1) * np.linalg.norm(pt[dst], axis=1) + 1e-12
    sim = np.clip(dot / norm, 0, 1)
    p = 1 - np.exp(-alpha * sim * np.sqrt(Ht[s] * Hs[s]) / np.clip(Ht[dst], 1e-8, None))
    return np.clip(p, 0, 1)

g_noloop = dgl.remove_self_loop(g.cpu())
edge_no_self = torch.stack(g_noloop.edges())
print(f"edges no_self {edge_no_self.shape[1]}")


edges no_self 491722


In [12]:
model = MLP().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=5e-4)

lam, tau, alpha, beta = 0.2, 0.9, 10, 0.5
best_val = 0
best_state = None
wait = 0
best_test = 0
teacher = t.detach()

for epoch in range(1, 501):
    model.train()

    if epoch == 1:
        cur = torch.stack([torch.arange(N), torch.arange(N)])
        w = torch.ones(N, 1)
    else:
        with torch.no_grad():
            probe = model(feats)
        a = alpha / (100 ** ((epoch - 1) / 499))
        p = hardness_scores(teacher, probe, edge_no_self, a)
        keep = torch.from_numpy(np.random.binomial(1, p).astype(bool))
        kept = edge_no_self[:, keep]
        self_loops = torch.stack([torch.arange(N), torch.arange(N)])
        cur = torch.cat([kept, self_loops], dim=1)
        w = torch.cat([
            torch.from_numpy(p[keep.numpy()]).float().unsqueeze(1),
            torch.ones(N, 1)
        ], dim=0)

    cur = cur.to(DEVICE)
    w = w.to(DEVICE)

    logits = model(feats)
    ce = F.cross_entropy(logits[idx_tr], labels[idx_tr])

    b = torch.from_numpy(np.random.beta(beta, beta, size=w.shape[0])).float().view(-1, 1).to(DEVICE)
    ew = w * b
    s_log = (logits / tau).log_softmax(1)[cur[0]]
    pt_s = (teacher / tau).softmax(1)[cur[0]]
    pt_d = (teacher / tau).softmax(1)[cur[1]]
    mix = pt_s * (1 - ew) + pt_d * ew
    mix /= mix.sum(1, keepdim=True).clamp_min(1e-12)
    t_log = (mix + 1e-12).log()
    kd = F.kl_div(s_log, t_log, reduction="batchmean", log_target=True)
    loss = lam * ce + (1 - lam) * kd

    opt.zero_grad()
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        lg = model(feats)
        v = accuracy(lg[idx_va], labels[idx_va])
        te = accuracy(lg[idx_te], labels[idx_te])

    if v > best_val:
        best_val = v
        best_test = te
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch % 40 == 0:
        print(f"{epoch} ce{ce.item():.3f} kd{kd.item():.3f} va{v:.4f} te{te:.4f} best{best_val:.4f}/{best_test:.4f} e{cur.shape[1]}")

    if wait >= 50:
        print(f"early {epoch}")
        break


40 ce0.414 kd0.419 va0.7067 te0.8020 best0.7400/0.7879 e421219
80 ce0.174 kd0.313 va0.8067 te0.8119 best0.8233/0.8138 e405691
120 ce0.153 kd0.298 va0.8133 te0.8061 best0.8400/0.8166 e388759
160 ce0.247 kd0.334 va0.7867 te0.8157 best0.8433/0.8102 e378122
200 ce0.146 kd0.279 va0.7933 te0.8152 best0.8533/0.8188 e346687
240 ce0.102 kd0.239 va0.8467 te0.8195 best0.8667/0.8200 e323314
early 260


In [13]:
model.load_state_dict(best_state)

with torch.no_grad():
    te = accuracy(model(feats)[idx_te], labels[idx_te])
    print(f"mixup {te:.4f} val {best_val:.4f}")

torch.save({"state": best_state}, OUT / "student_mlp_mixup.pt")
np.savez_compressed(OUT / "student_logits_mixup.npz", logits=model(feats).detach().cpu().numpy())


mixup 0.8200 val 0.8667
